# Local MIMIC-IV Hypotension POC: Three Cluster Explanation Views

This notebook runs locally, without Google Colab. It trains one small ViT proof-of-concept model, evaluates it, generates model explanations, and then plots three patient-cluster views:

1. **Explanation-derived clusters**: patients are clustered by model-importance maps.
2. **Value-derived clusters**: patients are clustered by binned clinical values and masks.
3. **Combined clusters**: patients are clustered by values, masks, and model-importance maps together.

All heatmaps use clinical value as the color. Black contours show high mean model importance for the patients in that cluster.

## 1. Project Paths And Imports

Run this notebook either from the repository root or from the `notebooks/` directory.

In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
PROJECT_DIR = cwd.parent if cwd.name == 'notebooks' else cwd
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RECORDS_PATH = PROJECT_DIR / 'data' / 'mimic_hypotension' / 'records.csv'
LABELS_PATH = PROJECT_DIR / 'data' / 'mimic_hypotension' / 'labels.csv'

print('PROJECT_DIR:', PROJECT_DIR)
print('RECORDS_PATH:', RECORDS_PATH, RECORDS_PATH.exists())
print('LABELS_PATH:', LABELS_PATH, LABELS_PATH.exists())

if not RECORDS_PATH.exists() or not LABELS_PATH.exists():
    raise FileNotFoundError('Expected records.csv and labels.csv under data/mimic_hypotension/.')

os.chdir(PROJECT_DIR)

## 2. Runtime Check

This uses the local Python environment. If an import fails, install dependencies with `pip install -r requirements.txt` and restart the kernel.

In [ ]:
import numpy as np
import pandas as pd
import torch

import interpretable_ts_vit

print('interpretable_ts_vit:', interpretable_ts_vit)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Create A Small Balanced Local POC Dataset

The full prepared MIMIC records table can be large. This cell creates a balanced subset for quick local experimentation. Set `USE_SUBSET = False` to use the full prepared records and labels.

In [ ]:
USE_SUBSET = True
PATIENTS_PER_CLASS = 120
RANDOM_STATE = 13

POC_DATA_DIR = PROJECT_DIR / 'data' / 'mimic_hypotension_local_poc'
POC_DATA_DIR.mkdir(parents=True, exist_ok=True)
POC_RECORDS_PATH = POC_DATA_DIR / 'records.csv'
POC_LABELS_PATH = POC_DATA_DIR / 'labels.csv'

if USE_SUBSET:
    labels = pd.read_csv(LABELS_PATH)
    counts = labels['label'].value_counts()
    per_class = min(PATIENTS_PER_CLASS, int(counts.min()))
    sampled_labels = (
        labels.groupby('label', group_keys=False)
        .sample(n=per_class, random_state=RANDOM_STATE)
        .sort_values('patient_id')
    )
    sampled_ids = set(sampled_labels['patient_id'].astype(str))
    sampled_labels.to_csv(POC_LABELS_PATH, index=False)

    first = True
    kept_records = 0
    for chunk in pd.read_csv(RECORDS_PATH, chunksize=500_000, dtype={'patient_id': str}):
        selected = chunk[chunk['patient_id'].isin(sampled_ids)]
        if len(selected):
            selected.to_csv(POC_RECORDS_PATH, index=False, mode='w' if first else 'a', header=first)
            first = False
            kept_records += len(selected)
    if first:
        pd.DataFrame(columns=['patient_id', 'variable', 'value', 'timestamp']).to_csv(POC_RECORDS_PATH, index=False)

    active_records_path = POC_RECORDS_PATH
    active_labels_path = POC_LABELS_PATH
    print(f'Created local POC subset: {len(sampled_labels)} patients, {kept_records} records')
else:
    active_records_path = RECORDS_PATH
    active_labels_path = LABELS_PATH

print('Active records:', active_records_path)
print('Active labels:', active_labels_path)
display(pd.read_csv(active_labels_path)['label'].value_counts().rename('count'))

## 4. Train, Save, Evaluate, And Explain Once

This run prepares tensors, trains the model, saves artifacts, evaluates the test split, and writes one explanation map per test patient. Clustering and plotting are handled in the next section so the same model/explanations can be reused for all three views.

In [ ]:
import shutil

from interpretable_ts_vit.config import Config, DataConfig, ModelConfig, TrainConfig, ExplainConfig, ClusterConfig
from interpretable_ts_vit.pipeline import PipelinePaths, PipelineRunConfig, run_pipeline

RUN_DIR = PROJECT_DIR / 'runs' / 'local_hypotension_three_views'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed_local_hypotension_three_views'
OVERWRITE_OUTPUTS = True

if OVERWRITE_OUTPUTS:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    shutil.rmtree(PROCESSED_DIR, ignore_errors=True)

config = Config(
    data=DataConfig(
        granularity='30min',
        aggregation='mean',
        val_fraction=0.2,
        test_fraction=0.2,
        random_state=RANDOM_STATE,
    ),
    model=ModelConfig(
        patch_size=(1, 4),
        embed_dim=48,
        depth=2,
        num_heads=4,
        mlp_ratio=2.0,
        dropout=0.1,
    ),
    train=TrainConfig(
        batch_size=32,
        epochs=3,
        learning_rate=1e-3,
        weight_decay=1e-4,
        device='cuda' if torch.cuda.is_available() else 'cpu',
    ),
    explain=ExplainConfig(method='grad_attention_rollout', target_class=1),
    cluster=ClusterConfig(n_clusters=4),
)

paths = PipelinePaths(
    records_path=active_records_path,
    labels_path=active_labels_path,
    processed_dir=PROCESSED_DIR,
    run_dir=RUN_DIR,
)

result = run_pipeline(
    PipelineRunConfig(
        paths=paths,
        config=config,
        prepare_mimic=False,
        prepare_tensors=True,
        train=True,
        evaluate=True,
        explain=True,
        cluster=False,
        plot=False,
        split='test',
    )
)

print('Artifacts:')
for key, value in result.artifacts.items():
    print(f'  {key}: {value}')
print('\nTrain metrics:', result.train_metrics)
print('\nEvaluation metrics:', result.evaluation_metrics)

## 5. Build Three Cluster Explanation Views

Each view clusters the same test patients differently, then plots cluster heatmaps. In every plot:

- color = mean observed clinical value
- gray = no observations for that cell
- black contour = high mean model importance

In [ ]:
import shutil
import numpy as np

from IPython.display import Image, display

from interpretable_ts_vit import TimeSeriesBinner
from interpretable_ts_vit.clustering import cluster_explanations
from interpretable_ts_vit.io import load_split
from interpretable_ts_vit.visualization import aggregate_cluster_value_matrices, plot_value_heatmap

N_CLUSTERS = 4
VALUE_WEIGHT = 1.0
EXPLANATION_WEIGHT = 1.0
MASK_WEIGHT = 0.25

binner = TimeSeriesBinner.load(RUN_DIR / 'binner.json')
test_dataset = load_split(RUN_DIR / 'test.npz')
explanation_dir = RUN_DIR / 'explanations' / 'test'


def load_cluster_importance_matrices(cluster_dir):
    matrices = {}
    for path in sorted(cluster_dir.glob('cluster_*.npy')):
        cluster = int(path.stem.split('_', 1)[1])
        matrices[cluster] = np.load(path)
    return matrices


def run_cluster_view(view_name, feature_mode, overlay_importance=True):
    view_dir = RUN_DIR / 'cluster_views' / view_name
    cluster_dir = view_dir / 'clusters'
    value_dir = view_dir / 'cluster_values'
    heatmap_dir = view_dir / 'heatmaps'
    shutil.rmtree(view_dir, ignore_errors=True)
    heatmap_dir.mkdir(parents=True, exist_ok=True)

    clustered = cluster_explanations(
        explanation_dir,
        n_clusters=N_CLUSTERS,
        output_dir=cluster_dir,
        dataset=test_dataset,
        feature_mode=feature_mode,
        value_weight=VALUE_WEIGHT,
        explanation_weight=EXPLANATION_WEIGHT,
        mask_weight=MASK_WEIGHT,
    )
    value_matrices = aggregate_cluster_value_matrices(test_dataset, clustered['assignments'], binner, output_dir=value_dir)
    importance_matrices = load_cluster_importance_matrices(cluster_dir)

    finite_values = [matrix[np.isfinite(matrix)] for matrix in value_matrices.values() if np.isfinite(matrix).any()]
    vmin = min(float(values.min()) for values in finite_values)
    vmax = max(float(values.max()) for values in finite_values)

    for cluster, value_matrix in value_matrices.items():
        plot_value_heatmap(
            value_matrix,
            binner.variable_vocab_,
            binner.time_bins_,
            heatmap_dir / f'cluster_{cluster}.png',
            title=f'{view_name}: cluster_{cluster}',
            vmin=vmin,
            vmax=vmax,
            importance_matrix=importance_matrices.get(cluster) if overlay_importance else None,
        )

    return {
        'view_name': view_name,
        'feature_mode': feature_mode,
        'cluster_dir': cluster_dir,
        'value_dir': value_dir,
        'heatmap_dir': heatmap_dir,
        'assignments': clustered['assignments'],
    }


views = [
    run_cluster_view('01_explanation_derived', 'explanation', overlay_importance=True),
    run_cluster_view('02_value_derived', 'value', overlay_importance=True),
    run_cluster_view('03_combined_value_importance', 'combined', overlay_importance=True),
]

for view in views:
    print(view['view_name'], '->', view['heatmap_dir'])
    display(view['assignments'].head())

## 6. Display Heatmaps For All Three Views

In [ ]:
for view in views:
    print('\n' + '=' * 90)
    print(view['view_name'])
    print('Cluster feature mode:', view['feature_mode'])
    print('Heatmaps:', view['heatmap_dir'])
    for heatmap in sorted(view['heatmap_dir'].glob('*.png')):
        print(heatmap.name)
        display(Image(filename=str(heatmap)))

## 7. Inspect Saved Evaluation Artifacts

In [ ]:
import json

metrics_path = RUN_DIR / 'test_evaluation_metrics.json'
predictions_path = RUN_DIR / 'test_predictions.csv'

if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
if predictions_path.exists():
    display(pd.read_csv(predictions_path).head())